In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
import warnings
warnings.filterwarnings('ignore')


# Airbnb vs Traditional Hotels: A Comparative Pricing Analysis

## Project Overview
This analysis compares two major accommodation platforms to identify specific pricing strategies, 
booking behaviors, and market positioning differences.

## Data Sources
1. **Hotel Booking Demand Dataset**
   - Source: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
   
2. **Airbnb Open Data**
   - Source: https://www.kaggle.com/datasets/arianazmoudeh/airbnbopendata


In [4]:
print("="*70)
print("LOADING DATASET 1: HOTEL BOOKINGS")
print("="*70)

df_hotel = pd.read_csv('data/hotel_bookings.csv')

print(f"\nDataset Shape: {df_hotel.shape}")
print(f"Rows: {df_hotel.shape[0]:,}")
print(f"Columns: {df_hotel.shape[1]}")

print(f"\nColumn Names:")
for i, col in enumerate(df_hotel.columns, 1):
    print(f"{i}. {col}")

print("\n" + "="*70)

LOADING DATASET 1: HOTEL BOOKINGS

Dataset Shape: (119390, 32)
Rows: 119,390
Columns: 32

Column Names:
1. hotel
2. is_canceled
3. lead_time
4. arrival_date_year
5. arrival_date_month
6. arrival_date_week_number
7. arrival_date_day_of_month
8. stays_in_weekend_nights
9. stays_in_week_nights
10. adults
11. children
12. babies
13. meal
14. country
15. market_segment
16. distribution_channel
17. is_repeated_guest
18. previous_cancellations
19. previous_bookings_not_canceled
20. reserved_room_type
21. assigned_room_type
22. booking_changes
23. deposit_type
24. agent
25. company
26. days_in_waiting_list
27. customer_type
28. adr
29. required_car_parking_spaces
30. total_of_special_requests
31. reservation_status
32. reservation_status_date



In [6]:
print("="*70)
print("LOADING DATASET 2: AIRBNB LISTINGS")
print("="*70)

df_airbnb = pd.read_csv('data/airbnb__data.csv')

print(f"\nDataset Shape: {df_airbnb.shape}")
print(f"Rows: {df_airbnb.shape[0]:,}")
print(f"Columns: {df_airbnb.shape[1]}")

print(f"\nColumn Names:")
for i, col in enumerate(df_airbnb.columns, 1):
    print(f"{i}. {col}")

print("\n" + "="*70)

LOADING DATASET 2: AIRBNB LISTINGS

Dataset Shape: (102599, 26)
Rows: 102,599
Columns: 26

Column Names:
1. id
2. NAME
3. host id
4. host_identity_verified
5. host name
6. neighbourhood group
7. neighbourhood
8. lat
9. long
10. country
11. country code
12. instant_bookable
13. cancellation_policy
14. room type
15. Construction year
16. price
17. service fee
18. minimum nights
19. number of reviews
20. last review
21. reviews per month
22. review rate number
23. calculated host listings count
24. availability 365
25. house_rules
26. license



In [7]:
print("="*70)
print("MISSING VALUES ANALYSIS - HOTELS")
print("="*70)

missing_hotel = df_hotel.isnull().sum()
missing_pct = (missing_hotel / len(df_hotel) * 100).round(2)

missing_df = pd.DataFrame({
    'Column': missing_hotel.index,
    'Missing_Count': missing_hotel.values,
    'Percentage': missing_pct.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df) > 0:
    print("\nColumns with missing values:\n")
    print(missing_df.to_string(index=False))
else:
    print("\nNo missing values detected!")

MISSING VALUES ANALYSIS - HOTELS

Columns with missing values:

  Column  Missing_Count  Percentage
 company         112593       94.31
   agent          16340       13.69
 country            488        0.41
children              4        0.00


In [8]:
print("="*70)
print("MISSING VALUES ANALYSIS - AIRBNB")
print("="*70)

missing_airbnb = df_airbnb.isnull().sum()
missing_pct_ab = (missing_airbnb / len(df_airbnb) * 100).round(2)

missing_df_ab = pd.DataFrame({
    'Column': missing_airbnb.index,
    'Missing_Count': missing_airbnb.values,
    'Percentage': missing_pct_ab.values
})

missing_df_ab = missing_df_ab[missing_df_ab['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df_ab) > 0:
    print("\nTop 20 Columns with missing values:\n")
    print(missing_df_ab.head(20).to_string(index=False))
else:
    print("\nNo missing values detected!")

MISSING VALUES ANALYSIS - AIRBNB

Top 20 Columns with missing values:

                        Column  Missing_Count  Percentage
                       license         102597      100.00
                   house_rules          52131       50.81
                   last review          15893       15.49
             reviews per month          15879       15.48
                       country            532        0.52
              availability 365            448        0.44
                     host name            406        0.40
                minimum nights            409        0.40
            review rate number            326        0.32
calculated host listings count            319        0.31
        host_identity_verified            289        0.28
                   service fee            273        0.27
                          NAME            250        0.24
                         price            247        0.24
             Construction year            214        0.21
 

In [9]:
# Data Cleaning Strategy
print("="*70)
print("DATA CLEANING APPROACH")
print("="*70)

print("\nHOTEL DATASET:")
print("Strategy: Remove or fill missing values based on context")
print("- High missing % columns (>40%): Drop if not essential")
print("- Numerical missing: Fill with 0 or median depending on meaning")
print("- Categorical missing: Fill with 'Unknown' or mode")
print("- Invalid data: Remove rows with impossible values")

print("\n\nAIRBNB DATASET:")
print("Strategy: Keep only essential columns with reasonable data completeness")
print("- Price column: Critical, remove rows if missing")
print("- Remove columns with >50% missing data")
print("- Clean price formatting if needed")
print("- Remove invalid prices (zero or negative)")

DATA CLEANING APPROACH

HOTEL DATASET:
Strategy: Remove or fill missing values based on context
- High missing % columns (>40%): Drop if not essential
- Numerical missing: Fill with 0 or median depending on meaning
- Categorical missing: Fill with 'Unknown' or mode
- Invalid data: Remove rows with impossible values


AIRBNB DATASET:
Strategy: Keep only essential columns with reasonable data completeness
- Price column: Critical, remove rows if missing
- Remove columns with >50% missing data
- Clean price formatting if needed
- Remove invalid prices (zero or negative)


In [10]:
# Clean Hotels Dataset
print("="*70)
print("CLEANING HOTEL DATASET")
print("="*70)

original_shape = df_hotel.shape
print(f"\nOriginal shape: {original_shape}")

print("\nCleaning steps:")

# Identify and drop high-missing columns
cols_to_drop = []
for col in df_hotel.columns:
    missing_pct = (df_hotel[col].isnull().sum() / len(df_hotel)) * 100
    if missing_pct > 40:
        cols_to_drop.append(col)
        print(f"- Dropping '{col}' ({missing_pct:.1f}% missing)")

if cols_to_drop:
    df_hotel = df_hotel.drop(cols_to_drop, axis=1)

# Fill missing values intelligently
for col in df_hotel.columns:
    missing_count = df_hotel[col].isnull().sum()
    if missing_count > 0:
        if df_hotel[col].dtype in ['float64', 'int64']:
            # For count-type numerical columns
            if any(keyword in col.lower() for keyword in ['children', 'babies', 'nights', 'guest', 'adults']):
                df_hotel[col].fillna(0, inplace=True)
                print(f"- Filled '{col}' with 0")
            else:
                median_val = df_hotel[col].median()
                df_hotel[col].fillna(median_val, inplace=True)
                print(f"- Filled '{col}' with median")
        else:
            # For categorical columns
            df_hotel[col].fillna('Unknown', inplace=True)
            print(f"- Filled '{col}' with 'Unknown'")

# Remove invalid price records
price_cols = [col for col in df_hotel.columns if any(x in col.lower() for x in ['adr', 'price', 'rate'])]
for price_col in price_cols:
    if price_col in df_hotel.columns:
        before = len(df_hotel)
        df_hotel = df_hotel[df_hotel[price_col] > 0]
        removed = before - len(df_hotel)
        if removed > 0:
            print(f"- Removed {removed:,} rows with invalid {price_col}")

print(f"\nFinal shape: {df_hotel.shape}")
print(f"Rows removed: {original_shape[0] - df_hotel.shape[0]:,}")
print(f"Remaining missing: {df_hotel.isnull().sum().sum()}")

CLEANING HOTEL DATASET

Original shape: (119390, 32)

Cleaning steps:
- Dropping 'company' (94.3% missing)
- Filled 'children' with 0
- Filled 'country' with 'Unknown'
- Filled 'agent' with median
- Removed 1,960 rows with invalid adr

Final shape: (117430, 31)
Rows removed: 1,960
Remaining missing: 0


In [11]:
# Clean Airbnb Dataset
print("="*70)
print("CLEANING AIRBNB DATASET")
print("="*70)

original_shape = df_airbnb.shape
print(f"\nOriginal shape: {original_shape}")

print("\nCleaning steps:")

# Find price column
price_col = None
for col in df_airbnb.columns:
    if 'price' in col.lower():
        price_col = col
        print(f"- Found price column: '{col}'")
        break

if price_col:
    # Remove missing prices
    before = len(df_airbnb)
    df_airbnb = df_airbnb[df_airbnb[price_col].notna()]
    removed = before - len(df_airbnb)
    if removed > 0:
        print(f"- Removed {removed:,} rows with missing prices")
    
    # Clean price if it's text
    if df_airbnb[price_col].dtype == 'object':
        print(f"- Cleaning price format")
        df_airbnb[price_col] = df_airbnb[price_col].astype(str).str.replace('$', '', regex=False)
        df_airbnb[price_col] = df_airbnb[price_col].str.replace(',', '', regex=False)
        df_airbnb[price_col] = df_airbnb[price_col].str.strip()
        df_airbnb[price_col] = pd.to_numeric(df_airbnb[price_col], errors='coerce')
        print(f"- Converted to numeric")
    
    # Remove invalid prices
    before = len(df_airbnb)
    df_airbnb = df_airbnb[df_airbnb[price_col] > 0]
    removed = before - len(df_airbnb)
    if removed > 0:
        print(f"- Removed {removed:,} rows with invalid prices")

# Remove high-missing columns
cols_before = len(df_airbnb.columns)
threshold = 0.5
cols_to_keep = []
for col in df_airbnb.columns:
    missing_pct = df_airbnb[col].isnull().sum() / len(df_airbnb)
    if missing_pct < threshold:
        cols_to_keep.append(col)

df_airbnb = df_airbnb[cols_to_keep]
cols_removed = cols_before - len(cols_to_keep)
print(f"- Removed {cols_removed} columns with >{threshold*100:.0f}% missing")

print(f"\nFinal shape: {df_airbnb.shape}")
print(f"Rows removed: {original_shape[0] - df_airbnb.shape[0]:,}")
print(f"Remaining missing: {df_airbnb.isnull().sum().sum()}")

CLEANING AIRBNB DATASET

Original shape: (102599, 26)

Cleaning steps:
- Found price column: 'price'
- Removed 247 rows with missing prices
- Cleaning price format
- Converted to numeric
- Removed 2 columns with >50% missing

Final shape: (102352, 24)
Rows removed: 247
Remaining missing: 35669


In [12]:
# Post-Cleaning Summary
print("="*70)
print("DATA CLEANING SUMMARY")
print("="*70)

print("\nHOTEL DATASET:")
print(f"Dimensions: {df_hotel.shape}")
print(f"Records: {df_hotel.shape[0]:,}")
print(f"Features: {df_hotel.shape[1]}")
print(f"Complete cases: {df_hotel.dropna().shape[0]:,}")
print(f"Completeness: {(df_hotel.dropna().shape[0]/df_hotel.shape[0]*100):.1f}%")

print("\n" + "-"*70)

print("\nAIRBNB DATASET:")
print(f"Dimensions: {df_airbnb.shape}")
print(f"Records: {df_airbnb.shape[0]:,}")
print(f"Features: {df_airbnb.shape[1]}")
print(f"Complete cases: {df_airbnb.dropna().shape[0]:,}")
print(f"Completeness: {(df_airbnb.dropna().shape[0]/df_airbnb.shape[0]*100):.1f}%")

print("\n" + "="*70)
print("Both datasets cleaned and ready for analysis")
print("="*70)

DATA CLEANING SUMMARY

HOTEL DATASET:
Dimensions: (117430, 31)
Records: 117,430
Features: 31
Complete cases: 117,430
Completeness: 100.0%

----------------------------------------------------------------------

AIRBNB DATASET:
Dimensions: (102352, 24)
Records: 102,352
Features: 24
Complete cases: 83,878
Completeness: 82.0%

Both datasets cleaned and ready for analysis
